# 05 — Early Detection from Prefix-Level Telemetry

## Research Question
Can attacks be detected before the final answer is produced?

## Hypothesis
Unsafe workflows leave partial telemetry traces that become observable before completion.

## Input Data
- `df_raw_all`
- `early_df_all` / prefix-level telemetry

## Prediction/Analysis Target
- `is_attack` and attack-specific labels at trace fractions 25%, 50%, 75%, 100%

## Validation Protocol
Leave-one-seed-out at each trace fraction; per-attack analysis and temporal gain.

## Expected Paper Artifact
- Early detection curves
- Attack-type × fraction heatmaps
- Temporal gain table


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import silhouette_score
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    make_scorer,
)

from evaluation import (
    load_jsonl, 
    evaluate_binary_prediction, 
    evaluate_multiclass_attack_type,
    fit_rf_feature_importance,
    evaluate_multiclass_feature_group_ablation,
    add_akc_phase_from_attack_type,
    normalize_seed_column,
    infer_telemetry_features,
    prepare_akc_phase_df,
    run_akc_phase_leave_one_seed,
    summarize_multiclass_results,
    collect_akc_phase_predictions_leave_one_seed,
    plot_confusion_matrix_from_predictions,
    infer_early_akc_features,
    classification_report,
    run_temporal_akc_phase_classification,
    plot_temporal_akc_curves,
    adversarial_fragmentation_index,
    risk_conditioned_fragmentation,
    semantic_fragmentation_proxy,
    add_benign_normalized_fragmentation,
    evaluate_leave_one_seed_out,
    TELEMETRY_NUMERIC,
    evaluate_feature_group_ablation,
    FEATURE_GROUPS,
    evaluate_leave_one_attack_out,
    compute_permutation_importance,
    run_operational_only_leave_one_seed,
    summarize_results,
    plot_exp1_operational_comparison,
    load_tamas_jsonl,
    build_early_df_from_agent_outputs,
    run_early_detection_leave_one_seed,
    plot_early_detection_curve,
    EARLY_FEATURES_CANDIDATES,
    validate_early_df_for_attack_analysis,
    run_early_detection_by_attack_type,
    summarize_early_detection_by_attack,
    plot_metric_curves_by_attack,
    compute_temporal_gain,
    infer_numeric_features,
    keep_existing_numeric,
    run_early_detection_feature_ablation,
    plot_ablation_curves,
    telemetry_guardrail_decision,
    run_akc_phase_leave_one_attack_type_out,
)
from datasets.tamas.tamas_features import build_all_feature_tables

RESULTS_DIR = Path("results/tamas")
RAW_DIR = RESULTS_DIR / "raw"
FEATURES_DIR = RESULTS_DIR / "features"
PLOTS_DIR = RESULTS_DIR / "plots"

scenario = "education"
architecture = "centralized_tamas"
CONDITIONS = [
    "benign",
    "byzantine",
    "colluding",
    "contradicting",
    "DPI",
    "impersonation",
    "IPI",
]
SEEDS = [1, 2, 3]
MODEL_NAMES = [
    "ticlazau/meta-llama-3.1-8b-instruct:latest",
    # "qwen3",
]

In [ ]:

# Load processed telemetry tables generated by 00_setup_and_feature_extraction.ipynb.
# If files are not available yet, run notebook 00 first.
PROCESSED_DIR = Path("results/tamas/processed")

for _name in ["episode_df_all", "agent_df_all", "early_df_all", "df_raw_all"]:
    _path = PROCESSED_DIR / f"{_name}.parquet"
    if _path.exists():
        globals()[_name] = pd.read_parquet(_path)
        print(f"Loaded {_name}: {globals()[_name].shape}")
    else:
        print(f"Missing {_path}; run 00_setup_and_feature_extraction.ipynb if this notebook needs it.")


### Experimento 4 — Early detection

Objetivo: avaliar se é possível detectar risco antes do fim do workflow.

In [ ]:
df_raw_lst = []
for seed in SEEDS:
    for llm_model in MODEL_NAMES:
        for condition in CONDITIONS:
            safe_model_name = llm_model.replace(":", "_").replace("/", "_")
            raw_path = str(RAW_DIR / f"tamas_{scenario}_{architecture}_{condition}_{safe_model_name}_seed_{seed}.jsonl")

            df_raw = load_tamas_jsonl(raw_path)
            df_raw_lst.append(df_raw)

df_raw_all = pd.concat(df_raw_lst, ignore_index=True)

In [ ]:
# inspect_tamas_trace_structure(df_raw_all)

early_df = build_early_df_from_agent_outputs(df_raw_all)

# print(early_df.shape)
# print(early_df.head())
# print(early_df["trace_fraction"].value_counts())
# print(early_df["is_attack"].value_counts(dropna=False))

# validate_early_detection_ready(early_df)

In [ ]:
# early_df_all = pd.concat(early_df_all)

# early_df_all["seed"] = [s[0] for s in early_df_all["seed"]]

early_results = run_early_detection_leave_one_seed(early_df)

early_summary = (
    early_results
    .groupby("trace_fraction")
    [["balanced_accuracy", "f1", "roc_auc", "auprc"]]
    .agg(["mean", "std"])
    .reset_index()
)

# early_summary.columns = [
#     "_".join(col).strip("_") if isinstance(col, tuple) else col
#     for col in early_summary.columns
# ]

# print(early_results)
# print(early_summary)

# early_results.to_csv("results/tamas/early_detection_prefix_raw.csv", index=False)
# early_summary.to_csv("results/tamas/early_detection_prefix_summary.csv", index=False)

In [ ]:
plot_early_detection_curve(early_summary, metric="roc_auc_mean")

In [ ]:
EARLY_FEATURES = [
    c for c in EARLY_FEATURES_CANDIDATES
    if c in early_df.columns
]

validate_early_df_for_attack_analysis(
    early_df,
    features=EARLY_FEATURES,
)

early_attack_results = run_early_detection_by_attack_type(
    early_df=early_df,
    features=EARLY_FEATURES,
    model_kind="rf",
    threshold=0.5,
)

# print(early_attack_results.shape)
# print(early_attack_results.head())

In [ ]:
# early_attack_results.to_csv(
#     OUTPUT_DIR / "early_detection_by_attack_type_raw.csv",
#     index=False,
# )
early_attack_summary = summarize_early_detection_by_attack(early_attack_results)

# print(early_attack_summary)

In [ ]:
# timing_classification = classify_detection_timing(
#     early_attack_summary,
#     metric="roc_auc_mean",
#     threshold=0.85,
# )

# print(timing_classification)

In [ ]:
roc_auc_pivot = early_attack_summary.pivot(
    index="attack_type",
    columns="trace_fraction",
    values="roc_auc_mean",
)

bacc_pivot = early_attack_summary.pivot(
    index="attack_type",
    columns="trace_fraction",
    values="balanced_accuracy_mean",
)

print("ROC-AUC by attack and trace fraction:")
print(roc_auc_pivot)

print("\nBalanced Accuracy by attack and trace fraction:")
print(bacc_pivot)

roc_auc_pivot.to_csv(OUTPUT_DIR / "roc_auc_by_attack_fraction.csv")
bacc_pivot.to_csv(OUTPUT_DIR / "balanced_accuracy_by_attack_fraction.csv")

In [ ]:
# plot_attack_fraction_heatmap(
#     roc_auc_pivot,
#     title="Early Detection by Attack Type: ROC-AUC",
#     output_path=OUTPUT_DIR / "heatmap_roc_auc_by_attack_fraction.png",
#     vmin=0.5,
#     vmax=1.0,
# )

# plot_attack_fraction_heatmap(
#     bacc_pivot,
#     title="Early Detection by Attack Type: Balanced Accuracy",
#     output_path=OUTPUT_DIR / "heatmap_balanced_accuracy_by_attack_fraction.png",
#     vmin=0.5,
#     vmax=1.0,
# )

In [ ]:
plot_metric_curves_by_attack(
    early_attack_summary,
    metric="roc_auc_mean",
    output_path=OUTPUT_DIR / "curves_by_attack_roc_auc.png",
)

plot_metric_curves_by_attack(
    early_attack_summary,
    metric="balanced_accuracy_mean",
    output_path=OUTPUT_DIR / "curves_by_attack_balanced_accuracy.png",
)

In [ ]:
temporal_gain = compute_temporal_gain(
    early_attack_summary,
    metric="roc_auc_mean",
)

# print(temporal_gain.sort_values("gain_25_to_100", ascending=False))

# temporal_gain.to_csv(
#     OUTPUT_DIR / "temporal_gain_by_attack.csv",
#     index=False,
# )

In [ ]:
# plot_temporal_gain(
#     temporal_gain,
#     gain_col="gain_25_to_100",
#     output_path=OUTPUT_DIR / "temporal_gain_25_to_100.png",
# )

# plot_temporal_gain(
#     temporal_gain,
#     gain_col="gain_50_to_75",
#     output_path=OUTPUT_DIR / "temporal_gain_50_to_75.png",
# )